# Something's Missing Here — train the siamese model on Colab

Two photos of a scene in, one boolean out: **did anything go missing?**

The flow of this notebook:

1. **Install** the few dependencies Colab doesn't ship with.
2. **Authenticate** — `HF_TOKEN` for the private dataset; optional `GH_TOKEN` for the private code repo.
3. **Download the dataset** from HF (`Shay85Gil/somethings_missing_here`).
4. **Get the training code** from the private GitHub repo (`ShezZai/Object_DeTaken`, any branch) — or upload the two scripts by hand.
5. **Configure** the run in a single JSON cell — every CLI flag of `train_missing.py` can be set there.
6. **Train**: scene-grouped 5-fold cross-validation. Whole scenes are held out per fold, so validation always asks "does this work in a place the model never saw?"
7. **Evaluate** on the held-out `test/` split (scenes that appear in no fold at all) with a verbose list of failed pairs.
8. **Predict** on a single before/after pair.

> **Tips**: open this notebook straight from the private repo via Colab's `File → Open notebook → GitHub` (tick *Include private repos*). And use a GPU runtime: `Runtime → Change runtime type → T4 GPU` — training is ~10–20 min on a T4 at 256², roughly 2× that at 384².

In [ ]:
%pip install -q "albumentations<2" timm keras huggingface_hub

## Authenticate

Two secrets, both best stored in Colab's secret manager (key icon in the left sidebar):

- **`HF_TOKEN`** (required) — Hugging Face token with read access; the dataset repo is private.
- **`GH_TOKEN`** (optional) — GitHub personal access token that can read the private `ShezZai/Object_DeTaken` repo, used to fetch the training code automatically. A fine-grained PAT with read-only *Contents* permission on that single repo is ideal (github.com → Settings → Developer settings → Fine-grained tokens). Without it, the next section falls back to a manual file-upload prompt.

In [ ]:
import os

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or __import__("getpass").getpass(
    "Hugging Face token (read access): ")

gh_token = secret("GH_TOKEN")
if gh_token:
    os.environ["GH_TOKEN"] = gh_token
    print("GitHub token loaded — code will be cloned automatically")
else:
    print("no GH_TOKEN secret — you will be prompted to upload the scripts")

## Download the dataset from Hugging Face

`snapshot_download` mirrors the dataset repo locally:

```
somethings_missing_here/
├── training/            286 pairs, 26 scenes (DeTaken + Remove360_based)
└── test/                47 pairs, 10 held-out scenes (same two collections)
```

Each pair folder holds `before.jpg`, `after.jpg`, and `label.json`
(`{"missing": true, "items": ["chair"]}` — or `"missing": false` for
no-change pairs).

In [ ]:
import pathlib
from huggingface_hub import snapshot_download

data_dir = pathlib.Path(snapshot_download(
    "Shay85Gil/somethings_missing_here", repo_type="dataset",
    local_dir="somethings_missing_here"))
print("dataset ready at", data_dir)

## Get the training code from GitHub

`train_missing.py` + `pairs_io.py` live in the private
`ShezZai/Object_DeTaken` GitHub repo — the dataset repo is data-only.
With a `GH_TOKEN` secret the next cell clones the repo and copies the two
scripts (the clone is deleted afterwards so the token lingers nowhere);
without one, it prompts for a manual upload.

Set `BRANCH` to try code from a feature branch or tag instead of `main` —
e.g. to test a training change before merging it. If the scripts are
already present from an earlier run, re-run with `FORCE = True` to fetch
the branch's version.

In [ ]:
import os, pathlib, shutil, subprocess

BRANCH = "main"     # any branch or tag of the repo
FORCE = False       # True: re-fetch even if the scripts are already present

SCRIPTS = ("train_missing.py", "pairs_io.py")
if not FORCE and all(pathlib.Path(f).exists() for f in SCRIPTS):
    print("scripts already present (set FORCE = True to re-fetch)")
elif os.environ.get("GH_TOKEN"):
    url = f"https://{os.environ['GH_TOKEN']}@github.com/ShezZai/Object_DeTaken.git"
    subprocess.run(["git", "clone", "--quiet", "--depth", "1",
                    "--branch", BRANCH, url, "_project"], check=True)
    for f in SCRIPTS:
        shutil.copy(f"_project/siamese_missing/{f}", f)
    shutil.rmtree("_project")   # keep the token out of any .git/config
    print(f"code cloned from GitHub (branch: {BRANCH})")
else:
    from google.colab import files
    print("please upload:",
          ", ".join(f for f in SCRIPTS if not pathlib.Path(f).exists()))
    files.upload()

## Configuration

One dict = the whole run. Keys mirror `train_missing.py`'s CLI flags (any flag works here); `"cmd"` picks the subcommand. Flags passed on the command line would override these values.

The interesting knobs:

| key | meaning |
|---|---|
| `keras` | `true` → Keras 3 + EfficientNetB0 (sets `KERAS_BACKEND=torch` behind the scenes); `false` → PyTorch + ResNet18 |
| `identity-negatives` | adds train-only "same photo twice" negatives, teaching *lighting change ≠ item gone* |
| `freeze-epochs` | epochs with the ImageNet encoder frozen (head-only warmup) before gentle fine-tuning |
| `patience` | early stop after N epochs without a new best validation AUC — the saved model is always the best epoch, not the last |
| `height`/`width` | 256 is fast; 384 catches smaller objects (~1 AUC point better, ~2× slower — drop `bs` to 8) |

In [ ]:
CONFIG = {
    "cmd": "train",
    "keras": True,
    "root": "somethings_missing_here/training",
    "cache": True,
    "identity-negatives": True,
    "folds": 5,
    "epochs": 60,
    "freeze-epochs": 10,
    "patience": 15,
    "height": 256,
    "width": 256,
    "bs": 16,
    "seed": 42,
    "prefix": "colab_fold_",
}

import json
with open("run_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
CONFIG

## Train

Reading the output, one line per fold:

```
fold 3:  63 val pairs | AUC 0.941 | acc 0.873 | TP 24 FP 6 FN 2 TN 31 | best epoch 55
```

- **AUC** — ranking quality across all thresholds (a random positive scores above a random negative that fraction of the time).
- **acc / TP / FP / FN / TN** — at the fold's own threshold (Youden's J on its validation split, stored in the checkpoint). FN = missed removals, FP = false alarms.
- **best epoch** — where validation AUC peaked; those weights are what got saved. Best epochs near the `epochs` cap mean training was still improving — raise `epochs`.
- The final `AUC m ± s` line: read the spread as well as the mean — at this data size one lucky fold can carry the average.

In [ ]:
!python train_missing.py --config run_config.json

## Evaluate on the held-out test split

These 10 scenes appear in **no** training fold, so this is the honest
generalization number. All 5 fold checkpoints vote as an ensemble (plus
horizontal-flip TTA), and every misclassified pair is listed —
`missed` = removal not flagged, `false alarm` = unchanged pair flagged.

The test split contains both classes (39 removals, 8 no-change pairs), so
AUC, precision, and false alarms are all measured for real.

In [ ]:
!python train_missing.py evaluate --keras --root somethings_missing_here/test \
    --ckpt "colab_fold_*.keras" --cache

## Predict on a single pair

Output: `prob` is the ensemble's probability that something went missing,
`threshold` the averaged per-fold decision cutoff, and `any_missing`
simply `prob > threshold`. `per_model` shows the 5 folds' individual
opinions — tight agreement means a trustworthy call, a wide spread means
the pair is genuinely ambiguous.

In [ ]:
!python train_missing.py predict --keras --ckpt "colab_fold_*.keras" \
    --before somethings_missing_here/test/DeTaken/test_michal_01/pair_01/before.jpg \
    --after  somethings_missing_here/test/DeTaken/test_michal_01/pair_01/after.jpg

## Keep the trained models

Colab wipes the VM when the runtime ends — zip the checkpoints and download
them (folder icon in the left sidebar → `checkpoints.zip`), or copy them to
Drive. Each fold is a `.keras` model plus a `.json` sidecar holding its
decision threshold — keep the two together.

In [ ]:
!zip -q checkpoints.zip colab_fold_*
!ls -lh checkpoints.zip